# CHCEŠ MYŠ? · 02 · ILUSTRACE KOCOUR

Samostatný obrazový běh pro kartu **Kocour**. Vstupní obrázek se načítá přímo z repozitáře `artmisanovak-prog/chces-mys`, cesta `doma/ilustrace/kocour.jpg`.

Tento notebook provádí pouze obrazovou fázi. Nepřidává pohádku ani samostatný textový vstup. Model je **Qwen3-VL-Embedding-2B**. Výstupem je embeddingový vektor a archiv s podklady a metadaty, nikoli slovní interpretace obrázku.

## 1. Připojení Google Drive a průběžné ukládání

Nejdřív se připojí Drive a vytvoří složka experimentu. Po každé dokončené fázi se důležité soubory kopírují na Drive. Pokud běh skončí chybou, uloží se také chybový záznam. Samotný výpočet embeddingu je jedna nedělitelná operace knihovny Qwen; její vnitřní procenta knihovna neposkytuje.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, time, zipfile, shutil, hashlib, platform, traceback
from datetime import datetime, timezone

PROJECT_DIR = Path('/content/CHCES_MYS_02_KOCOUR')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DIR = Path('/content/drive/MyDrive/CHCES_MYS_EXPERIMENTY/02_KOCOUR_IMAGE')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = PROJECT_DIR / "run_log.jsonl"

def log_event(event, **kwargs):
    record = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "event": event, **kwargs}
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    shutil.copy2(LOG_PATH, DRIVE_DIR / LOG_PATH.name)
    print("LOG:", event, kwargs if kwargs else "")

def checkpoint(*paths):
    saved = []
    for path in paths:
        path = Path(path)
        if path.exists():
            shutil.copy2(path, DRIVE_DIR / path.name)
            saved.append(path.name)
    print("ULOŽENO NA DRIVE:", ", ".join(saved) if saved else "zatím žádné soubory")

RUN_STARTED = datetime.now(timezone.utc).isoformat()
log_event("run_started", experiment="02_KOCOUR_IMAGE")
print("Pracovní složka:", PROJECT_DIR)
print("Složka na Google Drive:", DRIVE_DIR)
print("Průběžné ukládání: aktivní")

## 2. Stažení a příprava zdrojového obrázku

Notebook stáhne kocour.jpg z repozitáře artmisanovak-prog/chces-mys. Protože je tam obrázek vzhůru nohama, otočí se o 180° přímo při načtení. Původní soubor v GitHubu se nemění. Upravený pracovní vstup se hned uloží na Google Drive.

In [ ]:
import requests
from PIL import Image
from IPython.display import display

IMAGE_URL = "https://raw.githubusercontent.com/artmisanovak-prog/chces-mys/main/doma/ilustrace/kocour.jpg"
IMAGE_PATH = PROJECT_DIR / "kocour.jpg"

print("STAV 1/6: Stahuji zdrojový obrázek…")
response = requests.get(IMAGE_URL, timeout=60)
response.raise_for_status()
IMAGE_PATH.write_bytes(response.content)

print("STAV 2/6: Načítám obrázek a otáčím jej o 180°…")
img = Image.open(IMAGE_PATH).convert("RGB").rotate(180, expand=True)
img.save(IMAGE_PATH, format="JPEG", quality=95, subsampling=0)

display(img)
image_hash = hashlib.sha256(IMAGE_PATH.read_bytes()).hexdigest()
print("Použitý vstup:", IMAGE_PATH.name)
print("Rozměry po otočení:", img.size)
print("SHA-256 pracovního obrázku:", image_hash)
checkpoint(IMAGE_PATH)
log_event("image_prepared_and_saved", image="kocour.jpg", source_url=IMAGE_URL,
          dimensions=list(img.size), rotation_degrees=180, sha256=image_hash)

## 3. Experimentální instrukce

Jde pouze o obrazovou fázi. Text napsaný přímo v ilustraci zůstává součástí obrazu, není zadán jako samostatná pohádka. Instrukce odpovídá obrazovému běhu Myš a zachovává projektový rámec.

In [ ]:
INSTRUCTION = """Represent the visual artifact from the project "Chceš myš?" according to the
following project-specific conceptual framework. This is an image-only task.
Do not invent or import a story, a plot, or any text that was not supplied as
part of the image itself.

Within the illustration's conceptual system:
- The girl represents speech.
- The animal represents a disturbance or disruption of that speech.
- Their relation concerns language as represented through the visual artifact.
- The gap is not a mistake, missing information, or a simple mismatch.
- The gap is an active space in which interpretation can take place.
- This interpretive space can be approached from several points of view.
- These points of view have a circular topology, with relations such as
  adjacency and opposition. Do not assume or invent a fixed number of points
  of view, names, coordinates, or a complete topology when they have not
  been supplied.

Treat the handwritten wording visible in the illustration as part of the
visual artifact. Do not treat it as separate story text. Do not reduce the
task to identifying objects, matching labels, or deciding which other image
or text this illustration resembles.

Create a semantic representation of this illustration under these
instructions. Preserve ambiguity where the image does not determine a
single interpretation. Do not add a verbal explanation as a substitute
for the embedding."""

INSTRUCTION_PATH = PROJECT_DIR / "instructions.txt"
INSTRUCTION_PATH.write_text(INSTRUCTION, encoding="utf-8")
checkpoint(INSTRUCTION_PATH)
log_event("instruction_saved", characters=len(INSTRUCTION))
print(INSTRUCTION)
print("\nPočet znaků instrukce:", len(INSTRUCTION))

## 4. Instalace oficiální implementace Qwen

Notebook stáhne oficiální repozitář Qwen3-VL-Embedding a nainstaluje jej. Po dokončení vypíše stav. Pokud instalace selže, Colab zobrazí konkrétní chybu.

In [ ]:
import subprocess, sys, os

QWEN_REPO = Path('/content/Qwen3-VL-Embedding')
print("INSTALACE: připravuji oficiální repozitář Qwen3-VL-Embedding…")
try:
    if not QWEN_REPO.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/QwenLM/Qwen3-VL-Embedding.git", str(QWEN_REPO)],
            check=True
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(QWEN_REPO)],
        check=True
    )
    print("INSTALACE DOKONČENA:", QWEN_REPO)
    log_event("qwen_installation_complete", repository=str(QWEN_REPO))
except Exception as exc:
    log_event("qwen_installation_failed", error=repr(exc), traceback=traceback.format_exc())
    error_path = PROJECT_DIR / "error_installation.txt"
    error_path.write_text(traceback.format_exc(), encoding="utf-8")
    checkpoint(error_path, IMAGE_PATH, INSTRUCTION_PATH)
    raise

## 5. Automatická volba zařízení a načtení modelu

Zařízení se volí automaticky: CUDA, pokud ji Colab zpřístupní, jinak CPU. Tato volba není pevně nastavena. Notebook vypíše, které zařízení skutečně použil, a zaznamená začátek i konec načítání modelu.

In [ ]:
import torch
import sys
from pathlib import Path

QWEN_REPO = Path('/content/Qwen3-VL-Embedding')
if not (QWEN_REPO / 'src' / 'models' / 'qwen3_vl_embedding.py').exists():
    raise FileNotFoundError(f"Očekávaný soubor oficiální implementace nebyl nalezen: {QWEN_REPO / 'src/models/qwen3_vl_embedding.py'}")
sys.path.insert(0, str(QWEN_REPO))

print("STAV 3/6: Zjišťuji dostupné zařízení…")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Zvolené zařízení:", DEVICE)
print("CUDA dostupná:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

MODEL_NAME = "Qwen/Qwen3-VL-Embedding-2B"
log_event("model_loading_started", model=MODEL_NAME, device=DEVICE)
print("STAV 4/6: Načítám model", MODEL_NAME, "…")
print("Tato fáze může trvat déle; Colab během ní zobrazí běžící buňku.")
try:
    from src.models.qwen3_vl_embedding import Qwen3VLEmbedder
    model = Qwen3VLEmbedder(model_name_or_path=MODEL_NAME)
except Exception as exc:
    log_event("model_loading_failed", error=repr(exc), traceback=traceback.format_exc())
    error_path = PROJECT_DIR / "error_model_loading.txt"
    error_path.write_text(traceback.format_exc(), encoding="utf-8")
    checkpoint(error_path)
    raise

log_event("model_loading_complete", model=MODEL_NAME, device=DEVICE)
print("MODEL NAČTEN:", MODEL_NAME)

## 6. Výpočet obrazové reprezentace a okamžité uložení

Do modelu vstupuje jediný obrázek a projektová instrukce. Nejde o textovou ani multimodální fázi. Knihovna Qwen neposkytuje průběžné procento uvnitř model.process, proto notebook viditelně označí začátek a konec této operace. Jakmile se embedding vrátí, ihned se uloží na Google Drive, ještě před tvorbou metadat a ZIP archivu.

In [ ]:
import numpy as np

INSTRUCTION_PATH = PROJECT_DIR / "instructions.txt"
INSTRUCTION_PATH.write_text(INSTRUCTION, encoding="utf-8")
checkpoint(INSTRUCTION_PATH)
log_event("instruction_saved", characters=len(INSTRUCTION))

inputs = [{"image": str(IMAGE_PATH), "instruction": INSTRUCTION}]
log_event("embedding_started", model=MODEL_NAME, device=DEVICE, input_image="kocour.jpg")
print("STAV 5/6: MODEL PRÁVĚ POČÍTÁ OBRAZOVÝ EMBEDDING…")
print("Vstup: 1 obrázek + obrazová instrukce. Procentuální průběh této operace knihovna neposkytuje.")

try:
    embeddings = model.process(inputs)
    if hasattr(embeddings, "detach"):
        embedding = embeddings.detach().cpu().numpy()
    else:
        embedding = np.asarray(embeddings)
    embedding = np.asarray(embedding, dtype=np.float32)

    embedding_path = PROJECT_DIR / "kocour_image_embedding.npy"
    np.save(embedding_path, embedding)
    checkpoint(embedding_path)
    log_event("embedding_saved", shape=list(embedding.shape), dtype=str(embedding.dtype),
              norm=float(np.linalg.norm(embedding)))
    print("EMBEDDING VYPOČÍTÁN A IHNED ULOŽEN NA DRIVE.")
    print("Tvar:", embedding.shape)
    print("Datový typ:", embedding.dtype)
    print("Norma:", float(np.linalg.norm(embedding)))
except Exception as exc:
    log_event("embedding_failed", error=repr(exc), traceback=traceback.format_exc())
    error_path = PROJECT_DIR / "error_embedding.txt"
    error_path.write_text(traceback.format_exc(), encoding="utf-8")
    checkpoint(error_path, INSTRUCTION_PATH, IMAGE_PATH)
    raise

## 7. Metadata a kontrolní uložení

Metadata zaznamenají zdrojový obrázek, jeho otočení, přesnou instrukci, model, zařízení, verze knihoven, čas běhu a parametry embeddingu. V této fázi už je samotný embedding bezpečně uložen na Google Drive.

In [ ]:
metadata = {
    "project": "CHCEŠ MYŠ?",
    "experiment_id": "02",
    "phase": "image_only",
    "card": "kocour",
    "model": MODEL_NAME,
    "source_repository": "artmisanovak-prog/chces-mys",
    "source_path": "doma/ilustrace/kocour.jpg",
    "source_url": IMAGE_URL,
    "working_image_rotation_degrees": 180,
    "working_image_sha256": hashlib.sha256(IMAGE_PATH.read_bytes()).hexdigest(),
    "image_dimensions_after_rotation": list(img.size),
    "instruction": INSTRUCTION,
    "embedding_shape": list(embedding.shape),
    "embedding_dtype": str(embedding.dtype),
    "embedding_norm": float(np.linalg.norm(embedding)),
    "device": DEVICE,
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "cuda_available": torch.cuda.is_available(),
    "run_started_utc": RUN_STARTED,
    "run_finished_utc": datetime.now(timezone.utc).isoformat()
}
metadata_path = PROJECT_DIR / "metadata.json"
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
checkpoint(IMAGE_PATH, INSTRUCTION_PATH, embedding_path, metadata_path, LOG_PATH)
log_event("metadata_saved", metadata_file="metadata.json")
print("METADATA ULOŽENA.")
print("Obsah složky na Drive:", sorted(p.name for p in DRIVE_DIR.iterdir()))

## 8. Vytvoření ZIP archivu

ZIP obsahuje pracovní obrázek po otočení, přesnou instrukci, metadata, log běhu a embedding. Archiv se uloží na Google Drive. Pokud se běh přeruší před touto fází, jednotlivé dříve uložené soubory zůstávají ve složce experimentu na Drive.

In [ ]:
ZIP_NAME = "CHCES_MYS_KOCOUR_IMAGE_02.zip"
ZIP_PATH = Path('/content') / ZIP_NAME
files_to_archive = ["kocour.jpg", "instructions.txt", "metadata.json",
                    "kocour_image_embedding.npy", "run_log.jsonl"]
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for filename in files_to_archive:
        p = PROJECT_DIR / filename
        if p.exists():
            z.write(p, arcname=filename)

DRIVE_ZIP = DRIVE_DIR / ZIP_NAME
shutil.copy2(ZIP_PATH, DRIVE_ZIP)
log_event("archive_created", archive=ZIP_NAME, size_bytes=ZIP_PATH.stat().st_size)
print("STAV 6/6: ZIP VYTVOŘEN.")
print("Archiv na Google Drive:", DRIVE_ZIP)
print("Velikost ZIP (MB):", round(ZIP_PATH.stat().st_size / 1024**2, 3))
print("Soubory v archivu:", files_to_archive)

## 9. Stažení ZIP do telefonu

Po dokončení všech předchozích fází tato buňka nabídne stažení archivu. Kopie ZIP i jednotlivé výsledky zůstávají na Google Drive.

In [ ]:
from google.colab import files
files.download(str(ZIP_PATH))

## Poznámka k výsledku

Embedding je číselná reprezentace vytvořená modelem podle zadané obrazové instrukce. Není to slovní interpretace ani důkaz, že model správně pochopil jednotlivé pojmy. Notebook uchovává surový vektor, vstupní obrázek, přesnou instrukci, metadata a log, aby bylo možné výsledek později analyzovat a porovnávat.